In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.preprocessing import MinMaxScaler
from itertools import count
import random
from collections import deque, namedtuple
import pandas as pd
import numpy as np
import math

from forexenv import ForexTradingEnv #type: ignore

In [2]:
data = pd.read_csv('/content/dados.csv')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [3]:
def normalize_data(data):
    """
    Normaliza os dados usando média e desvio padrão.
    """
    return (data - data.mean()) / data.std()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [4]:
columns = ['volume', 'RSI', '%K', '%D']
data[columns] = data[columns].apply(normalize_data)

In [5]:
class DQN(nn.Module):
    """Rede Neural Profunda para a DQN (Deep Q-Network)"""

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Primeira camada totalmente conectada (input para hidden)
        self.layer1 = nn.Linear(n_observations, 128)
        # Segunda camada totalmente conectada (hidden para hidden)
        self.layer2 = nn.Linear(128, 128)
        # Terceira camada totalmente conectada (hidden para output)
        self.layer3 = nn.Linear(128, n_actions)

    # Chamado com um elemento para determinar a próxima ação, ou um lote durante a otimização
    # Retorna tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        # Aplicar a função de ativação ReLU após a primeira camada
        x = F.relu(self.layer1(x))
        # Aplicar a função de ativação ReLU após a segunda camada
        x = F.relu(self.layer2(x))
        # Passar pela terceira camada sem ativação para obter os valores Q para cada ação
        return self.layer3(x)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [6]:
BATCH_SIZE = 128 # BATCH_SIZE é o número de transições amostradas do buffer de reprodução
GAMMA = 0.99     # GAMMA é o fator de desconto
EPS_START = 0.9  # EPS_START é o valor inicial do épsilon
EPS_END = 0.05   # EPS_END é o valor final do épsilon
EPS_DECAY = 1000 # EPS_DECAY controla a taxa de decaimento exponencial do épsilon, maior significa um decaimento mais lento
TAU = 0.005      # TAU é a taxa de atualização da rede alvo
LR = 1e-4        # LR é a taxa de aprendizado do otimizador ``AdamW``

In [7]:
env = ForexTradingEnv(df=data)
n_actions = env.action_space.n # Obtenha o número de ações
state = env.reset()            # Obtenha o número de observações de estado
n_observations = len(state)

In [8]:
# Se a GPU for usada
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [9]:
# Cria a rede neural de política e move para o dispositivo especificado (CPU ou GPU)
policy_net = DQN(n_observations, n_actions).to(device)

# Cria a rede neural alvo e move para o dispositivo especificado
target_net = DQN(n_observations, n_actions).to(device)

# Inicializa a rede neural alvo com os mesmos pesos da rede de política
target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [10]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

In [11]:
class ReplayMemory(object):
    """Classe para armazenar e amostrar transições de memória de repetição"""

    def __init__(self, capacity):
        # Inicializa uma deque (fila de duas extremidades) com um tamanho máximo de `capacity`
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Salva uma transição"""
        # Adiciona uma nova transição à memória
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Amostra um lote de transições aleatoriamente"""
        # Retorna um número especificado (batch_size) de transições aleatórias da memória
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """Retorna o tamanho atual da memória"""
        # Retorna o número de transições atualmente armazenadas na memória
        return len(self.memory)

In [12]:
BUFFER_SIZE = int(1e5)  # Tamanho do buffer de replay
memory = ReplayMemory(BUFFER_SIZE) # Inicializa a memória de replay com a capacidade especificada

# Inicializa o otimizador AdamW com os parâmetros da policy_net, taxa de aprendizado LR, e amsgrad habilitado
# amsgrad=True: Habilita a variante AMSGrad do Adam, que pode proporcionar maior estabilidade em alguns casos
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)

In [13]:
steps_done = 0  # Inicializa o contador de passos

def select_action(state):
    global steps_done  # Usa a variável global steps_done para manter o controle do número de passos realizados
    sample = random.random()  # Gera um número aleatório entre 0 e 1
    # Calcula o valor de epsilon (probabilidade de escolher uma ação aleatória)
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1  # Incrementa o contador de passos

    # Verifica se o número aleatório é maior que o epsilon
    if sample > eps_threshold:
        # Se o número aleatório for maior que o epsilon, selecione a ação com maior recompensa esperada (ação "greedy")
        with torch.no_grad():
            # policy_net(state) retorna um tensor de valores Q para cada ação
            # t.max(1) retorna o valor máximo em cada linha, max(1) retorna (valores, indices)
            # .indices.view(1, 1) retorna o índice da ação com maior valor Q
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        # Caso contrário, selecione uma ação aleatória
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [14]:
def optimize_model():
    # Verifica se há transições suficientes no replay memory para formar um batch
    if len(memory) < BATCH_SIZE:
        return  # Se não houver, retorna sem fazer nada

    # Amostra um batch de transições do replay memory
    transitions = memory.sample(BATCH_SIZE)

    # Transpõe o batch para obter um Transition de batch-arrays em vez de um batch-array de Transitions
    # Lembre-se que Transition tem forma: Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))
    # Isso facilita o trabalho com os dados em PyTorch
    batch = Transition(*zip(*transitions))

    # Cria uma máscara de estados não finais e concatena os elementos do batch
    # Um estado final é aquele em que a simulação terminou
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Computa Q(s_t, a) - o modelo computa Q(s_t), então selecionamos as colunas das ações tomadas
    # Estas são as ações que teriam sido tomadas para cada estado do batch de acordo com policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Computa os retornos para todos os próximos estados.
    # Valores esperados das ações para non_final_next_states são computados com  base no target_net "mais antigo"
    # Selecionando a melhor recompensa com max(1).values
    # Isso é mesclado com base na máscara, de modo que teremos o valor do estado esperado ou 0 caso o estado fosse final
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    # Computa os valores Q esperados
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Computa a perda Huber
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    ################# Otimiza o modelo #################
    optimizer.zero_grad()  # Zera os gradientes acumulados
    loss.backward()        # Computa os gradientes
    # Clipping dos gradientes para evitar explosão de gradientes
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()       # Atualiza os pesos do modelo

In [15]:
%%time
num_episodes = 10

rewards = []

for i_episode in range(num_episodes):
    # Inicializa o ambiente e obtém o estado inicial
    reward_episode = 0
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        # Seleciona uma ação com base no estado atual usando a política epsilon-greedy
        action = select_action(state)
        observation, reward, terminated, info = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated

        if terminated:
            next_state = None  # Define próximo estado como None se o episódio terminou
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward) # Armazena a transição na memória

        state = next_state # Move para o próximo estado

        optimize_model() # Executa uma etapa de otimização
        reward_episode += reward.item()

        # Atualização suave dos pesos da rede alvo
        # θ′ ← τ θ + (1 − τ)θ′
        # Atualiza suavemente os pesos da rede alvo com base nos pesos da rede de política
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()

        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)

        target_net.load_state_dict(target_net_state_dict) # Carrega os novos pesos suavizados de volta na rede alvo

        if done:
          break
    rewards.append(reward_episode)
    print(f'Episode: {i_episode+1}:')
    print(f'Reward: {reward_episode} | Total Rewards {sum(rewards)}')
    print("###################################################################")

print(f'Complete!\n Total Rewards: {sum(rewards)}')

Episode: 1:
Reward: -3935461.0 | Total Rewards -3935461.0
###################################################################
Episode: 2:
Reward: -3380920.0 | Total Rewards -7316381.0
###################################################################
Episode: 3:
Reward: 3131630.0 | Total Rewards -4184751.0
###################################################################
Episode: 4:
Reward: 3617709.0 | Total Rewards -567042.0
###################################################################
Episode: 5:
Reward: 3589913.0 | Total Rewards 3022871.0
###################################################################
Episode: 6:
Reward: 3325530.0 | Total Rewards 6348401.0
###################################################################
Episode: 7:
Reward: 3514301.0 | Total Rewards 9862702.0
###################################################################
Episode: 8:
Reward: 1976294.0 | Total Rewards 11838996.0
###################################################################
Ep